# FT-NCFM single-task LIBERO A100 gate

This notebook first reproduces the sign/ranking/normalization proxy gate, then trains and evaluates five downstream policies on one official LIBERO task. Demonstrations are partitioned by trajectory: 35 train, 5 influence reference, 5 selection validation, and 5 final test. The simulator uses five fixed official initial states. This is a one-task proof of concept, not a four-suite paper replication.

In [ ]:
REPO_URL = "https://github.com/hppddub/NCFM.git"
UPSTREAM_URL = "https://github.com/gszfwsb/NCFM.git"
BRANCH = "codex/ft-ncfm-replication"
REPO_DIR = "/content/NCFM"
RESULT_ROOT = "/content/ft-ncfm-results"
LIBERO_ROOT = "/content/LIBERO"


In [ ]:
import csv, io, subprocess, sys
query = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], text=True).strip()
gpu, memory_mib, driver = next(csv.reader(io.StringIO(query), skipinitialspace=True))
assert "A100" in gpu.upper(), f"Expected A100, got {gpu}"
print({"gpu": gpu, "memory_mib": int(memory_mib), "driver": driver})


In [ ]:
import os
from pathlib import Path
repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", BRANCH], check=True)
if "upstream" not in subprocess.check_output(["git", "-C", str(repo), "remote"], text=True).split():
    subprocess.run(["git", "-C", str(repo), "remote", "add", "upstream", UPSTREAM_URL], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "--no-tags", "upstream", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.0", "torchvision==0.20.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "PyYAML==6.0.2", "pytest>=8.3", "ruff>=0.8", "h5py>=3.11,<4"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(repo)], check=True)
os.chdir(repo)


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "ruff", "check", "ft_ncfm", "tests", "infra"], cwd=repo, check=True)


In [ ]:
RUN_PROXY_ABLATION = True
if RUN_PROXY_ABLATION:
    subprocess.run([sys.executable, "infra/colab/run_experiment.py", "--config", "configs/ft_ncfm/minivla_factorial_ablation.yaml", "--persistent-root", RESULT_ROOT, "--run-name", "factorial-ablation", "--entry-module", "ft_ncfm.ablation", "--seeds", "42", "--require-gpu", "--require-a100"], cwd=repo, check=True)


In [ ]:
subprocess.run([sys.executable, "infra/colab/bootstrap_libero.py", "--root", LIBERO_ROOT], cwd=repo, check=True)


In [ ]:
subprocess.run([sys.executable, "infra/colab/run_experiment.py", "--config", "configs/ft_ncfm/libero_spatial_task0.yaml", "--persistent-root", RESULT_ROOT, "--run-name", "libero-spatial-task0", "--entry-module", "ft_ncfm.libero_experiment", "--seeds", "42", "--require-gpu", "--require-a100"], cwd=repo, check=True)


In [ ]:
import json
summary_path = Path(RESULT_ROOT) / "libero-spatial-task0" / "aggregate_summary.json"
print(json.dumps(json.loads(summary_path.read_text()), indent=2))


In [ ]:
import hashlib, shutil
archive = shutil.make_archive("/content/ft-ncfm-libero-spatial-task0", "zip", Path(RESULT_ROOT) / "libero-spatial-task0")
payload = Path(archive).read_bytes()
print({"archive": archive, "bytes": len(payload), "sha256": hashlib.sha256(payload).hexdigest()})
from google.colab import files
files.download(archive)
